# 61 — M5 independent one-step verifier

Read-only verification of `one_step_certificate/`.
Final arithmetic is reimplemented in `src/independent_one_step_verifier.py` and must not
import `src.certificate` / `src.influence`.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('pytest') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pytest>=8'])
import pytest
print('pytest:', pytest.__version__)

In [ ]:
from pathlib import Path
import os, sys

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'validated_4d_su2_rg_codex_design.md').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('M5 project root was not found.')
PERSIST_ROOT = Path(os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
run_id = os.environ.get('VALIDATED_RG_M5_RUN_ID', 'M5-20260720T051020Z-c3800fceaa80')
PACKAGE = PERSIST_ROOT / 'runs' / run_id / 'artifacts' / 'one_step_certificate'
print('PACKAGE =', PACKAGE)

In [ ]:
from src.common import atomic_write_json
from src.independent_one_step_verifier import verify_one_step_package

if not PACKAGE.is_dir():
    raise RuntimeError(
        'one_step_certificate package is absent. '
        'Run notebook 60 after open proof obligations are closed, '
        'or use cpu_fixture_cert mode in tests.'
    )
REPORT = verify_one_step_package(PACKAGE)
out = PERSIST_ROOT / 'runs' / run_id / 'reports' / 'M5_independent_verifier_report.json'
atomic_write_json(out, REPORT.payload())
{
    'agreement': REPORT.agreement,
    'independent_verdict': REPORT.independent_verdict,
    'package_manifest_hash': REPORT.package_manifest_hash,
    'report_path': str(out),
}